In [14]:
def load_real_documents(self):
    """Charge les vrais documents du CNRS"""
    import glob
    from langchain.document_loaders import (
        PyPDFLoader,
        BSHTMLLoader,
        UnstructuredFileLoader
    )

    documents = []

    # Charger PDFs
    pdf_files = glob.glob(f"{self.data_path}/*.pdf")
    for pdf_file in pdf_files:
        loader = PyPDFLoader(pdf_file)
        docs = loader.load()
        documents.extend(docs)

    # Charger HTMLs
    html_files = glob.glob(f"{self.data_path}/*.html")
    for html_file in html_files:
        loader = BSHTMLLoader(html_file)
        docs = loader.load()
        documents.extend(docs)

    return documents

In [15]:
# -*- coding: utf-8 -*-
"""Évaluation de modèles RAG pour le Hackathon CNRS - Concours Ingénieurs

Ce notebook permet d'évaluer et comparer différents modèles pour l'agent conversationnel.
"""

# ============================================================================
# 1. INSTALLATION DES LIBRAIRIES
# ============================================================================
print("Installation des librairies nécessaires...")

!pip install -q langchain langchain-community chromadb pypdf sentence-transformers
!pip install -q "unstructured[all-docs]" beautifulsoup4 lxml
!pip install -q ragas evaluate openai tiktoken
!pip install -q datasets pandas matplotlib seaborn plotly
!pip install -q --upgrade protobuf

# ============================================================================
# 2. IMPORT DES LIBRAIRIES
# ============================================================================
import os
import json
import pandas as pd
import numpy as np
from typing import List, Dict, Any, Tuple
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Librairies d'évaluation
from datasets import Dataset
import evaluate
from ragas import evaluate as ragas_evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
    answer_correctness,
    answer_similarity
)

# LangChain
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.schema import Document

# Modèles
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("✓ Librairies importées avec succès!")

# ============================================================================
# 3. CONFIGURATION
# ============================================================================
class Config:
    """Configuration du projet"""
    # Chemins
    DATA_PATH = "/content/cnrs_data"
    VECTOR_DB_PATH = "/content/vector_db"
    RESULTS_PATH = "/content/results"

    # Modèles à comparer
    EMBEDDING_MODELS = [
        "sentence-transformers/all-MiniLM-L6-v2",  # Léger et rapide
        "intfloat/multilingual-e5-large",  # Multilingue performant
        "dangvantuan/sentence-camembert-large"  # Français optimisé
    ]

    LLM_MODELS = [
        {"name": "mistral-7b", "model_id": "mistralai/Mistral-7B-Instruct-v0.1"},
        {"name": "zephyr-7b", "model_id": "HuggingFaceH4/zephyr-7b-beta"},
        {"name": "llama2-7b", "model_id": "meta-llama/Llama-2-7b-chat-hf"}
    ]

    # Paramètres RAG
    CHUNK_SIZE = 512
    CHUNK_OVERLAP = 50
    TOP_K_RETRIEVAL = 3

    # Paramètres d'évaluation
    TEST_QUESTIONS = [
        # Questions factuelles sur concours spécifiques
        {
            "question": "Quelles sont les conditions d'accès au concours d'ingénieur de recherche en informatique?",
            "reference_answer": "À vérifier dans les documents",
            "category": "factual",
            "difficulty": "medium"
        },
        {
            "question": "Quel est le salaire d'un ingénieur d'étude débutant au CNRS?",
            "reference_answer": "À vérifier dans les documents",
            "category": "factual",
            "difficulty": "easy"
        },
        # Questions comparatives
        {
            "question": "Quelle est la différence entre ingénieur de recherche et ingénieur d'étude?",
            "reference_answer": "À vérifier dans les documents",
            "category": "comparative",
            "difficulty": "medium"
        },
        # Questions d'orientation
        {
            "question": "À quel concours dois-je postuler si j'ai un master en biologie et 3 ans d'expérience?",
            "reference_answer": "À vérifier dans les documents",
            "category": "orientation",
            "difficulty": "hard"
        },
        # Questions hors domaine
        {
            "question": "Quelle est la capitale de l'Italie?",
            "reference_answer": "hors_domaine",
            "category": "out_of_scope",
            "difficulty": "easy"
        },
        # Questions ambigües
        {
            "question": "Comment se préparer au concours?",
            "reference_answer": "À vérifier dans les documents",
            "category": "ambiguous",
            "difficulty": "medium"
        }
    ]

    @staticmethod
    def create_directories():
        """Crée les répertoires nécessaires"""
        os.makedirs(Config.DATA_PATH, exist_ok=True)
        os.makedirs(Config.VECTOR_DB_PATH, exist_ok=True)
        os.makedirs(Config.RESULTS_PATH, exist_ok=True)

Config.create_directories()

# ============================================================================
# 4. CLASSE POUR CHARGER ET TRAITER LES DOCUMENTS
# ============================================================================
class DocumentProcessor:
    """Classe pour charger et traiter les documents CNRS"""

    def __init__(self, data_path: str):
        self.data_path = data_path
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=Config.CHUNK_SIZE,
            chunk_overlap=Config.CHUNK_OVERLAP,
            length_function=len,
            separators=["\n\n", "\n", " ", ""]
        )

    def load_sample_documents(self):
        """Charge des documents d'exemple (à adapter avec vos vrais documents)"""
        print("Chargement des documents d'exemple...")

        # Documents factices pour démonstration
        sample_docs = [
            Document(
                page_content="""Concours Ingénieur de Recherche en Informatique - 2025
                Conditions d'accès : Master ou diplôme équivalent + 4 ans d'expérience.
                Salaire brut mensuel débutant : 2800€.
                Lieu d'affectation : Toulouse, Paris, Lyon.
                Date limite de candidature : 15 mars 2025.
                Épreuves : écrit (algorithmique), oral (projet technique).""",
                metadata={"source": "concours_001.html", "type": "concours", "domaine": "informatique"}
            ),
            Document(
                page_content="""Concours Ingénieur d'Étude en Biologie - 2025
                Conditions d'accès : Licence + 2 ans d'expérience.
                Salaire brut mensuel débutant : 2400€.
                Lieu : Montpellier, Strasbourg.
                Avantages : mutuelle, titre de transport, 45 jours de congés.""",
                metadata={"source": "concours_002.html", "type": "concours", "domaine": "biologie"}
            ),
            Document(
                page_content="""Différence entre Ingénieur de Recherche (IR) et Ingénieur d'Étude (IE) :
                IR : conception et direction de projets de recherche, grade A.
                IE : réalisation technique sous supervision, grade B.
                Salaire IR > Salaire IE d'environ 400€ brut mensuel.""",
                metadata={"source": "guide_carriere.pdf", "type": "guide", "page": "12"}
            ),
            Document(
                page_content="""Procédure de concours CNRS :
                1. Dépôt de candidature en ligne
                2. Présélection sur dossier
                3. Épreuves écrites (si applicable)
                4. Audition devant jury
                5. Publication des résultats sous 30 jours.
                Les candidats étrangers doivent justifier de leur droit au travail en France.""",
                metadata={"source": "procedure_concours.pdf", "type": "procedure", "version": "2025"}
            )
        ]

        print(f"✓ {len(sample_docs)} documents chargés")
        return sample_docs

    def split_documents(self, documents: List[Document]) -> List[Document]:
        """Découpe les documents en chunks"""
        print("Découpage des documents...")
        chunks = self.text_splitter.split_documents(documents)
        print(f"✓ {len(chunks)} chunks créés")
        return chunks

# ============================================================================
# 5. CLASSE POUR LA BASE DE DONNÉES VECTORIELLE
# ============================================================================
class VectorStoreManager:
    """Gère les bases de données vectorielles pour différents modèles d'embedding"""

    def __init__(self):
        self.vector_stores = {}

    def create_vector_store(self, chunks: List[Document], embedding_model_name: str,
                           store_name: str = "default"):
        """Crée une base de données vectorielle"""
        print(f"Création de la vector store avec {embedding_model_name}...")

        embeddings = HuggingFaceEmbeddings(
            model_name=embedding_model_name,
            model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
            encode_kwargs={'normalize_embeddings': True}
        )

        vector_store = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            persist_directory=f"{Config.VECTOR_DB_PATH}/{store_name}",
            collection_name=store_name
        )

        self.vector_stores[store_name] = vector_store
        print(f"✓ Vector store '{store_name}' créée avec {len(chunks)} documents")
        return vector_store

    def get_retriever(self, store_name: str, top_k: int = Config.TOP_K_RETRIEVAL):
        """Récupère un retriever pour une vector store"""
        if store_name not in self.vector_stores:
            raise ValueError(f"Vector store '{store_name}' non trouvée")

        return self.vector_stores[store_name].as_retriever(
            search_kwargs={"k": top_k}
        )

# ============================================================================
# 6. CLASSE POUR LES MODÈLES LLM
# ============================================================================
class LLMManager:
    """Gère différents modèles LLM"""

    def __init__(self):
        self.models = {}
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Utilisation du device: {self.device}")

    def load_model(self, model_config: Dict[str, str]):
        """Charge un modèle LLM"""
        model_name = model_config["name"]
        model_id = model_config["model_id"]

        print(f"Chargement du modèle {model_name} ({model_id})...")

        try:
            # Tokenizer
            tokenizer = AutoTokenizer.from_pretrained(model_id)
            tokenizer.pad_token = tokenizer.eos_token

            # Modèle
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
                device_map="auto" if self.device == "cuda" else None,
                load_in_8bit=True if self.device == "cuda" else False
            )

            # Pipeline
            pipe = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=True,
                top_p=0.95,
                repetition_penalty=1.15
            )

            # LangChain wrapper
            llm = HuggingFacePipeline(pipeline=pipe)

            self.models[model_name] = {
                "llm": llm,
                "tokenizer": tokenizer,
                "model": model
            }

            print(f"✓ Modèle {model_name} chargé avec succès")

        except Exception as e:
            print(f"✗ Erreur lors du chargement de {model_name}: {str(e)}")
            return None

    def get_qa_chain(self, model_name: str, retriever, prompt_template: str = None):
        """Crée une chaîne QA pour un modèle"""
        if model_name not in self.models:
            raise ValueError(f"Modèle {model_name} non chargé")

        if prompt_template is None:
            prompt_template = """Tu es un assistant expert sur les concours CNRS.
            Utilise le contexte suivant pour répondre à la question.
            Si tu ne sais pas ou si l'information n'est pas dans le contexte, dis-le clairement.
            Fournis des réponses précises et cite tes sources.

            Contexte: {context}

            Question: {question}

            Réponse:"""

        prompt = PromptTemplate(
            template=prompt_template,
            input_variables=["context", "question"]
        )

        qa_chain = RetrievalQA.from_chain_type(
            llm=self.models[model_name]["llm"],
            chain_type="stuff",
            retriever=retriever,
            chain_type_kwargs={"prompt": prompt},
            return_source_documents=True
        )

        return qa_chain

# ============================================================================
# 7. CLASSE D'ÉVALUATION
# ============================================================================
class RAGEvaluator:
    """Évalue les performances des modèles RAG"""

    def __init__(self, test_questions: List[Dict]):
        self.test_questions = test_questions
        self.results = []

    def generate_test_dataset(self, qa_chain, model_name: str, embedding_name: str):
        """Génère les réponses pour le jeu de test"""
        print(f"Génération des réponses pour {model_name}/{embedding_name}...")

        data_records = []

        for i, q in enumerate(self.test_questions):
            try:
                # Génération de la réponse
                result = qa_chain({"query": q["question"]})

                # Extraction du contexte
                context = "\n\n".join([doc.page_content for doc in result["source_documents"]])

                # Enregistrement des résultats
                record = {
                    "question": q["question"],
                    "answer": result["result"],
                    "contexts": [context],
                    "ground_truth": q.get("reference_answer", ""),
                    "category": q.get("category", "unknown"),
                    "difficulty": q.get("difficulty", "medium"),
                    "model": model_name,
                    "embedding": embedding_name,
                    "timestamp": datetime.now().isoformat(),
                    "source_documents": [
                        {
                            "content": doc.page_content[:200] + "...",
                            "source": doc.metadata.get("source", "unknown")
                        }
                        for doc in result["source_documents"]
                    ]
                }

                data_records.append(record)

                print(f"  Question {i+1}/{len(self.test_questions)}: ✓")

            except Exception as e:
                print(f"  Question {i+1}: ✗ Erreur: {str(e)}")
                record = {
                    "question": q["question"],
                    "answer": f"ERREUR: {str(e)}",
                    "contexts": [""],
                    "ground_truth": q.get("reference_answer", ""),
                    "category": q.get("category", "unknown"),
                    "model": model_name,
                    "embedding": embedding_name
                }
                data_records.append(record)

        return Dataset.from_list(data_records)

    def evaluate_with_ragas(self, dataset):
        """Évalue avec les métriques RAGAS"""
        print("Évaluation avec RAGAS...")

        try:
            result = ragas_evaluate(
                dataset,
                metrics=[
                    faithfulness,  # Les faits sont-ils basés sur le contexte ?
                    answer_relevancy,  # La réponse est-elle pertinente à la question ?
                    context_recall,  # Tout le contexte pertinent est-il récupéré ?
                    context_precision,  # Le contexte récupéré est-il pertinent ?
                    answer_correctness,  # La réponse est-elle correcte vs vérité terrain ?
                ]
            )

            return result

        except Exception as e:
            print(f"✗ Erreur RAGAS: {str(e)}")
            return None

    def custom_evaluation(self, dataset):
        """Évaluation personnalisée avec métriques spécifiques"""
        print("Évaluation personnalisée...")

        metrics = {
            "answer_length": [],
            "refusal_rate": [],
            "citation_presence": [],
            "hallucination_score": [],
            "relevance_score": []
        }

        for record in dataset:
            answer = record["answer"]

            # Longueur de la réponse
            metrics["answer_length"].append(len(answer))

            # Taux de refus
            refusal_keywords = ["je ne sais pas", "pas dans", "information non disponible",
                              "ne peux pas répondre", "hors de mon domaine"]
            is_refusal = any(keyword in answer.lower() for keyword in refusal_keywords)
            metrics["refusal_rate"].append(1 if is_refusal else 0)

            # Présence de citations
            has_citation = "source" in answer.lower() or "document" in answer.lower() or "concours" in answer
            metrics["citation_presence"].append(1 if has_citation else 0)

            # Score d'hallucination (simplifié)
            # Ici on pourrait utiliser un LLM pour évaluer
            metrics["hallucination_score"].append(0.5)  # Placeholder

            # Score de pertinence (simplifié)
            question_words = set(record["question"].lower().split())
            answer_words = set(answer.lower().split())
            if question_words:
                overlap = len(question_words.intersection(answer_words)) / len(question_words)
                metrics["relevance_score"].append(min(overlap * 2, 1.0))
            else:
                metrics["relevance_score"].append(0)

        # Calcul des moyennes
        avg_metrics = {}
        for metric_name, values in metrics.items():
            if values:
                avg_metrics[f"custom_{metric_name}"] = np.mean(values)
            else:
                avg_metrics[f"custom_{metric_name}"] = 0

        return avg_metrics

    def run_evaluation(self, qa_chain, model_name: str, embedding_name: str):
        """Exécute l'évaluation complète"""
        print(f"\n{'='*60}")
        print(f"ÉVALUATION: {model_name} + {embedding_name}")
        print(f"{'='*60}")

        # Génération du dataset
        dataset = self.generate_test_dataset(qa_chain, model_name, embedding_name)

        # Évaluation RAGAS
        ragas_results = self.evaluate_with_ragas(dataset)

        # Évaluation personnalisée
        custom_metrics = self.custom_evaluation(dataset)

        # Combinaison des résultats
        evaluation_result = {
            "model": model_name,
            "embedding": embedding_name,
            "timestamp": datetime.now().isoformat(),
            "num_questions": len(self.test_questions),
            "dataset": dataset
        }

        if ragas_results:
            evaluation_result["ragas_metrics"] = ragas_results
            # Extraction des scores moyens
            for metric_name, score in ragas_results.items():
                evaluation_result[f"ragas_{metric_name}"] = float(score)

        evaluation_result.update(custom_metrics)

        # Calcul du score composite (pondéré)
        weights = {
            "ragas_faithfulness": 0.3,  # Importance critique pour votre cas
            "ragas_answer_relevancy": 0.2,
            "custom_refusal_rate": 0.2,  # Important pour éviter les hallucinations
            "ragas_answer_correctness": 0.2,
            "custom_citation_presence": 0.1
        }

        composite_score = 0
        weight_sum = 0

        for metric, weight in weights.items():
            if metric in evaluation_result:
                composite_score += evaluation_result[metric] * weight
                weight_sum += weight

        if weight_sum > 0:
            composite_score = composite_score / weight_sum

        evaluation_result["composite_score"] = composite_score
        evaluation_result["safety_score"] = (
            evaluation_result.get("ragas_faithfulness", 0) * 0.7 +
            evaluation_result.get("custom_refusal_rate", 0) * 0.3
        )

        self.results.append(evaluation_result)

        print(f"\nRésumé des scores pour {model_name}/{embedding_name}:")
        print(f"  • Score composite: {composite_score:.3f}")
        print(f"  • Score sécurité: {evaluation_result['safety_score']:.3f}")
        print(f"  • Faithfulness: {evaluation_result.get('ragas_faithfulness', 'N/A'):.3f}")
        print(f"  • Taux de refus: {evaluation_result.get('custom_refusal_rate', 'N/A'):.3f}")

        return evaluation_result

# ============================================================================
# 8. VISUALISATION DES RÉSULTATS
# ============================================================================
class ResultsVisualizer:
    """Visualise les résultats d'évaluation"""

    @staticmethod
    def plot_comparison(results: List[Dict]):
        """Crée des visualisations comparatives"""

        # Préparation des données
        df_data = []
        for result in results:
            df_data.append({
                "Model": result["model"],
                "Embedding": result["embedding"],
                "Composite Score": result.get("composite_score", 0),
                "Safety Score": result.get("safety_score", 0),
                "Faithfulness": result.get("ragas_faithfulness", 0),
                "Answer Relevancy": result.get("ragas_answer_relevancy", 0),
                "Refusal Rate": result.get("custom_refusal_rate", 0),
                "Citation Presence": result.get("custom_citation_presence", 0)
            })

        df = pd.DataFrame(df_data)

        # 1. Heatmap des scores
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))

        # Score composite
        axes[0,0].bar(range(len(df)), df["Composite Score"])
        axes[0,0].set_title("Score Composite par Configuration")
        axes[0,0].set_xticks(range(len(df)))
        axes[0,0].set_xticklabels([f"{m}/{e}" for m, e in zip(df["Model"], df["Embedding"].str.split("/").str[-1])],
                                 rotation=45)
        axes[0,0].set_ylim(0, 1)

        # Score de sécurité
        axes[0,1].bar(range(len(df)), df["Safety Score"], color='red')
        axes[0,1].set_title("Score de Sécurité")
        axes[0,1].set_xticks(range(len(df)))
        axes[0,1].set_xticklabels(df["Model"], rotation=45)
        axes[0,1].set_ylim(0, 1)

        # Métriques principales
        metrics_to_plot = ["Faithfulness", "Answer Relevancy", "Refusal Rate"]
        x = range(len(df))
        width = 0.25

        for i, metric in enumerate(metrics_to_plot):
            axes[1,0].bar([pos + i*width for pos in x], df[metric],
                         width=width, label=metric)

        axes[1,0].set_title("Métriques Détaillées")
        axes[1,0].set_xticks([pos + width for pos in x])
        axes[1,0].set_xticklabels(df["Model"], rotation=45)
        axes[1,0].legend()
        axes[1,0].set_ylim(0, 1)

        # Heatmap
        heatmap_data = df[["Composite Score", "Safety Score", "Faithfulness",
                          "Answer Relevancy", "Refusal Rate"]].T
        im = axes[1,1].imshow(heatmap_data.values, cmap='YlOrRd', aspect='auto')
        axes[1,1].set_title("Heatmap des Performances")
        axes[1,1].set_yticks(range(len(heatmap_data)))
        axes[1,1].set_yticklabels(heatmap_data.index)
        axes[1,1].set_xticks(range(len(df)))
        axes[1,1].set_xticklabels(df["Model"], rotation=45)

        plt.colorbar(im, ax=axes[1,1])
        plt.tight_layout()
        plt.savefig(f"{Config.RESULTS_PATH}/comparison_plot.png", dpi=300, bbox_inches='tight')
        plt.show()

        # 2. Radar chart
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, polar=True)

        metrics_radar = ["Faithfulness", "Answer Relevancy", "Refusal Rate",
                        "Citation Presence", "Composite Score"]

        angles = np.linspace(0, 2*np.pi, len(metrics_radar), endpoint=False).tolist()
        angles += angles[:1]  # Fermer le cercle

        for idx, row in df.iterrows():
            values = [row[m] for m in metrics_radar]
            values += values[:1]  # Fermer le cercle
            ax.plot(angles, values, 'o-', linewidth=2,
                   label=f"{row['Model']}/{row['Embedding'].split('/')[-1][:10]}")
            ax.fill(angles, values, alpha=0.1)

        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(metrics_radar)
        ax.set_ylim(0, 1)
        ax.set_title("Radar Chart des Performances", size=15, y=1.1)
        ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
        plt.savefig(f"{Config.RESULTS_PATH}/radar_chart.png", dpi=300, bbox_inches='tight')
        plt.show()

        return df

    @staticmethod
    def generate_report(results: List[Dict], df_comparison: pd.DataFrame):
        """Génère un rapport détaillé"""

        report = {
            "timestamp": datetime.now().isoformat(),
            "total_configurations": len(results),
            "best_configuration": None,
            "best_safety_configuration": None,
            "detailed_results": [],
            "recommendations": []
        }

        # Trouver les meilleures configurations
        if len(results) > 0:
            best_comp = max(results, key=lambda x: x.get("composite_score", 0))
            best_safety = max(results, key=lambda x: x.get("safety_score", 0))

            report["best_configuration"] = {
                "model": best_comp["model"],
                "embedding": best_comp["embedding"],
                "composite_score": best_comp.get("composite_score", 0),
                "safety_score": best_comp.get("safety_score", 0)
            }

            report["best_safety_configuration"] = {
                "model": best_safety["model"],
                "embedding": best_safety["embedding"],
                "composite_score": best_safety.get("composite_score", 0),
                "safety_score": best_safety.get("safety_score", 0)
            }

        # Résultats détaillés
        for result in results:
            report["detailed_results"].append({
                "model": result["model"],
                "embedding": result["embedding"],
                "scores": {
                    k: v for k, v in result.items()
                    if k in ["composite_score", "safety_score", "ragas_faithfulness",
                            "ragas_answer_relevancy", "custom_refusal_rate"]
                }
            })

        # Recommandations
        if report["best_configuration"]:
            report["recommendations"] = [
                f"Configuration recommandée: {report['best_configuration']['model']} "
                f"avec {report['best_configuration']['embedding'].split('/')[-1]}",
                f"Score composite: {report['best_configuration']['composite_score']:.3f}",
                f"Pour la sécurité maximale: {report['best_safety_configuration']['model']} "
                f"({report['best_safety_configuration']['safety_score']:.3f})",
                "Prioriser faithfulness et refusal rate pour éviter les hallucinations",
                "Toujours inclure la citation des sources dans le prompt"
            ]

        # Sauvegarde du rapport
        report_path = f"{Config.RESULTS_PATH}/evaluation_report.json"
        with open(report_path, 'w', encoding='utf-8') as f:
            json.dump(report, f, indent=2, ensure_ascii=False)

        # Génération d'un résumé texte
        summary_path = f"{Config.RESULTS_PATH}/summary.txt"
        with open(summary_path, 'w', encoding='utf-8') as f:
            f.write("="*60 + "\n")
            f.write("RAPPORT D'ÉVALUATION - AGENT CONVERSATIONNEL CNRS\n")
            f.write("="*60 + "\n\n")

            f.write("CONFIGURATION OPTIMALE:\n")
            f.write(f"  Modèle: {report['best_configuration']['model']}\n")
            f.write(f"  Embedding: {report['best_configuration']['embedding']}\n")
            f.write(f"  Score: {report['best_configuration']['composite_score']:.3f}\n\n")

            f.write("CONFIGURATION LA PLUS SÉCURISÉE:\n")
            f.write(f"  Modèle: {report['best_safety_configuration']['model']}\n")
            f.write(f"  Score sécurité: {report['best_safety_configuration']['safety_score']:.3f}\n\n")

            f.write("RECOMMANDATIONS:\n")
            for rec in report["recommendations"]:
                f.write(f"  • {rec}\n")

        print(f"\n✓ Rapport généré: {report_path}")
        print(f"✓ Résumé généré: {summary_path}")

        return report

# ============================================================================
# 9. PIPELINE PRINCIPALE
# ============================================================================
def main_pipeline():
    """Pipeline principale d'évaluation"""

    print("="*60)
    print("DÉMARRAGE DE L'ÉVALUATION DES MODÈLES RAG")
    print("="*60)

    # Initialisation
    doc_processor = DocumentProcessor(Config.DATA_PATH)
    vector_manager = VectorStoreManager()
    llm_manager = LLMManager()
    evaluator = RAGEvaluator(Config.TEST_QUESTIONS)
    visualizer = ResultsVisualizer()

    # 1. Chargement des documents
    print("\n1. Chargement des documents...")
    documents = doc_processor.load_sample_documents()
    chunks = doc_processor.split_documents(documents)

    # 2. Tests avec quelques combinaisons (pour rapidité)
    # Dans un vrai hackathon, tester toutes les combinaisons

    test_combinations = [
        {"embedding": Config.EMBEDDING_MODELS[0], "llm": Config.LLM_MODELS[0]},
        {"embedding": Config.EMBEDDING_MODELS[2], "llm": Config.LLM_MODELS[0]},
        {"embedding": Config.EMBEDDING_MODELS[0], "llm": Config.LLM_MODELS[1]},
    ]

    all_results = []

    for i, combo in enumerate(test_combinations):
        print(f"\n{'#'*60}")
        print(f"COMBINAISON {i+1}/{len(test_combinations)}")
        print(f"Embedding: {combo['embedding']}")
        print(f"LLM: {combo['llm']['name']}")
        print(f"{'#'*60}")

        try:
            # Création de la vector store
            store_name = f"store_{i}"
            vector_store = vector_manager.create_vector_store(
                chunks,
                combo["embedding"],
                store_name
            )

            # Chargement du modèle LLM
            llm_manager.load_model(combo["llm"])

            # Création de la chaîne QA
            retriever = vector_manager.get_retriever(store_name)
            qa_chain = llm_manager.get_qa_chain(combo["llm"]["name"], retriever)

            # Évaluation
            result = evaluator.run_evaluation(
                qa_chain,
                combo["llm"]["name"],
                combo["embedding"]
            )

            all_results.append(result)

        except Exception as e:
            print(f"✗ Erreur avec la combinaison {i+1}: {str(e)}")
            continue

    # 3. Visualisation et rapport
    if all_results:
        print("\n" + "="*60)
        print("GÉNÉRATION DES VISUALISATIONS ET DU RAPPORT")
        print("="*60)

        # Préparation des données pour visualisation
        df_comparison = visualizer.plot_comparison(all_results)

        # Génération du rapport
        report = visualizer.generate_report(all_results, df_comparison)

        print("\n" + "="*60)
        print("ÉVALUATION TERMINÉE AVEC SUCCÈS!")
        print("="*60)

        # Affichage des meilleurs résultats
        print("\n🏆 MEILLEURES CONFIGURATIONS:")
        print(f"1. Optimale: {report['best_configuration']['model']} "
              f"({report['best_configuration']['composite_score']:.3f})")
        print(f"2. Sécurité: {report['best_safety_configuration']['model']} "
              f"({report['best_safety_configuration']['safety_score']:.3f})")

        return all_results, report
    else:
        print("✗ Aucun résultat d'évaluation disponible")
        return None, None

# ============================================================================
# 10. EXÉCUTION
# ============================================================================
if __name__ == "__main__":
    # Pour Google Colab, vérifier la disponibilité GPU
    print(f"GPU disponible: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"Mémoire GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} Go")

    # Exécution de la pipeline
    results, report = main_pipeline()

    # Sauvegarde des résultats bruts
    if results:
        results_path = f"{Config.RESULTS_PATH}/raw_results.json"
        with open(results_path, 'w', encoding='utf-8') as f:
            json.dump([r for r in results if 'dataset' in r],
                     f, indent=2, default=str, ensure_ascii=False)
        print(f"\n✓ Résultats bruts sauvegardés: {results_path}")

# ============================================================================
# 11. FONCTIONS UTILITAIRES SUPPLÉMENTAIRES
# ============================================================================
def analyze_specific_question(results: List[Dict], question_index: int = 0):
    """Analyse détaillée d'une question spécifique"""
    if not results:
        print("Aucun résultat disponible")
        return

    print("\n" + "="*60)
    print(f"ANALYSE DÉTAILLÉE - QUESTION {question_index + 1}")
    print("="*60)

    question_text = Config.TEST_QUESTIONS[question_index]["question"]
    print(f"Question: {question_text}")
    print(f"Catégorie: {Config.TEST_QUESTIONS[question_index]['category']}")
    print(f"Difficulté: {Config.TEST_QUESTIONS[question_index]['difficulty']}")
    print()

    for result in results:
        if 'dataset' in result:
            dataset = result['dataset']
            if question_index < len(dataset):
                record = dataset[question_index]
                print(f"\n[{result['model']}/{result['embedding'].split('/')[-1]}]")
                print(f"Réponse: {record['answer'][:200]}...")
                print(f"Sources utilisées: {len(record['source_documents'])}")
                for doc in record['source_documents'][:2]:
                    print(f"  • {doc['source']}: {doc['content']}")

def test_custom_question(qa_chain, question: str):
    """Test une question personnalisée"""
    print(f"\nQuestion: {question}")
    result = qa_chain({"query": question})
    print(f"\nRéponse: {result['result']}")
    print(f"\nSources utilisées:")
    for i, doc in enumerate(result["source_documents"], 1):
        print(f"{i}. [{doc.metadata.get('source', 'Unknown')}]")
        print(f"   {doc.page_content[:150]}...")

# Exemple d'utilisation après exécution:
# Pour analyser une question spécifique:
# analyze_specific_question(results, question_index=0)

# Pour tester une question personnalisée (si une chaîne QA est disponible):
# test_custom_question(qa_chain, "Quel est le salaire d'un ingénieur débutant?")

Installation des librairies nécessaires...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 0.0.2 requires numpy<2,>=1, but you have numpy 2.2.6 which is incompatible.
langchain-community 0.0.13 requires numpy<2,>=1, but you have numpy 2.2.6 which is incompatible.
langchain 0.1.0 requires numpy<2,>=1, but you have numpy 2.2.6 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
xarray 2025.12.0 requires packaging>=24.1, but you have packaging 23.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 r

PydanticUserError: The `__modify_schema__` method is not supported in Pydantic v2. Use `__get_pydantic_json_schema__` instead in class `SecretStr`.

For further information visit https://errors.pydantic.dev/2.12/u/custom-json-schema

In [ ]:
"""
Système d'évaluation complet pour modèles LLM/RAG - Hackathon SID 2025
Projet: Agent conversationnel pour les concours ingénieurs CNRS

Ce notebook permet de:
1. Charger et préparer les documents (HTML, PDF)
2. Créer un système RAG avec différents modèles LLM
3. Évaluer et comparer les performances avec plusieurs métriques
4. Générer des rapports de comparaison

Installation des dépendances nécessaires
"""

# =====================================================================
# SECTION 1: INSTALLATION DES BIBLIOTHÈQUES
# =====================================================================

# Installation avec versions spécifiques pour éviter les conflits
!pip install -q langchain==0.1.0
!pip install -q langchain-community==0.0.13
!pip install -q langchain-openai==0.0.2
!pip install -q langchain-anthropic==0.1.1
!pip install -q chromadb==0.4.22
!pip install -q sentence-transformers==2.2.2
!pip install -q ragas==0.1.4
!pip install -q datasets==2.16.1
!pip install -q pypdf==3.17.4
!pip install -q beautifulsoup4==4.12.3
!pip install -q html2text==2020.1.16
!pip install -q unstructured==0.11.8
!pip install -q openai==1.7.2
!pip install -q anthropic==0.8.1
!pip install -q pandas matplotlib seaborn plotly
!pip install -q openpyxl

print("✅ Installation terminée!")

# =====================================================================
# SECTION 2: IMPORTS ET CONFIGURATION
# =====================================================================

import os
import json
import pandas as pd
import numpy as np
from typing import List, Dict, Any
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# LangChain - imports corrigés
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, UnstructuredHTMLLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.schema import Document

# LangChain LLMs
try:
    from langchain_openai import ChatOpenAI
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False
    print("⚠️ langchain-openai non disponible")

try:
    from langchain_anthropic import ChatAnthropic
    ANTHROPIC_AVAILABLE = True
except ImportError:
    ANTHROPIC_AVAILABLE = False
    print("⚠️ langchain-anthropic non disponible")

# RAGAS pour l'évaluation
try:
    from ragas import evaluate
    from ragas.metrics import (
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    )
    from datasets import Dataset
    RAGAS_AVAILABLE = True
except ImportError:
    RAGAS_AVAILABLE = False
    print("⚠️ RAGAS non disponible - évaluation limitée")

# Configuration de base
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Imports chargés avec succès!")
if OPENAI_AVAILABLE:
    print("  ✓ OpenAI disponible")
if ANTHROPIC_AVAILABLE:
    print("  ✓ Anthropic disponible")
if RAGAS_AVAILABLE:
    print("  ✓ RAGAS disponible")

# =====================================================================
# SECTION 3: CONFIGURATION DES CLÉS API
# =====================================================================

"""
IMPORTANT: Configurer vos clés API
Vous devez avoir accès à au moins un fournisseur de LLM
"""

def setup_api_keys():
    """Configure les clés API de manière interactive"""

    openai_key = None
    anthropic_key = None

    # Méthode 1: Essayer les secrets Colab
    try:
        from google.colab import userdata
        try:
            openai_key = userdata.get('OPENAI_API_KEY')
            print("✅ Clé OpenAI trouvée dans les secrets Colab")
        except:
            pass

        try:
            anthropic_key = userdata.get('ANTHROPIC_API_KEY')
            print("✅ Clé Anthropic trouvée dans les secrets Colab")
        except:
            pass
    except:
        print("ℹ️  Pas dans Google Colab")

    # Méthode 2: Saisie manuelle si nécessaire
    if not openai_key and not anthropic_key:
        print("\n⚠️ Aucune clé API trouvée dans les secrets Colab")
        print("Vous pouvez :")
        print("  1. Ajouter vos clés dans les secrets Colab (🔑 dans la barre latérale)")
        print("  2. Les entrer maintenant (moins sécurisé)\n")

        choice = input("Voulez-vous entrer vos clés maintenant ? (o/n): ").lower()

        if choice == 'o':
            openai_key = input("Clé OpenAI (ou Entrée pour ignorer): ").strip()
            anthropic_key = input("Clé Anthropic (ou Entrée pour ignorer): ").strip()

            if not openai_key:
                openai_key = None
            if not anthropic_key:
                anthropic_key = None

    # Configuration des variables d'environnement
    if openai_key:
        os.environ["OPENAI_API_KEY"] = openai_key
        print("✅ Clé OpenAI configurée")

    if anthropic_key:
        os.environ["ANTHROPIC_API_KEY"] = anthropic_key
        print("✅ Clé Anthropic configurée")

    if not openai_key and not anthropic_key:
        print("\n❌ ERREUR: Aucune clé API configurée!")
        print("Le système ne pourra pas fonctionner sans au moins une clé API.")
        return False

    return True

# Exécuter la configuration
API_CONFIGURED = setup_api_keys()

# =====================================================================
# SECTION 4: CHARGEMENT ET PRÉPARATION DES DOCUMENTS
# =====================================================================

class DocumentLoader:
    """Classe pour charger et préparer les documents CNRS"""

    def __init__(self, data_path: str = "./data"):
        """
        Initialise le chargeur de documents

        Args:
            data_path: Chemin vers le dossier contenant les documents
        """
        self.data_path = Path(data_path)
        self.documents = []

    def load_html_files(self, html_folder: str) -> List:
        """
        Charge tous les fichiers HTML (fiches de concours)

        Args:
            html_folder: Dossier contenant les fichiers HTML

        Returns:
            Liste de documents chargés
        """
        html_path = self.data_path / html_folder
        docs = []

        if not html_path.exists():
            print(f"⚠️ Dossier {html_path} non trouvé")
            return docs

        for html_file in html_path.glob("*.html"):
            try:
                loader = UnstructuredHTMLLoader(str(html_file))
                doc = loader.load()
                # Ajouter des métadonnées
                doc[0].metadata['source_type'] = 'concours'
                doc[0].metadata['filename'] = html_file.name
                docs.extend(doc)
                print(f"✅ Chargé: {html_file.name}")
            except Exception as e:
                print(f"❌ Erreur avec {html_file.name}: {e}")

        return docs

    def load_pdf_files(self, pdf_folder: str = ".") -> List:
        """
        Charge tous les fichiers PDF (documents généraux)

        Args:
            pdf_folder: Dossier contenant les PDF

        Returns:
            Liste de documents chargés
        """
        pdf_path = self.data_path / pdf_folder
        docs = []

        if not pdf_path.exists():
            print(f"⚠️ Dossier {pdf_path} non trouvé")
            return docs

        for pdf_file in pdf_path.glob("*.pdf"):
            try:
                loader = PyPDFLoader(str(pdf_file))
                doc = loader.load()
                # Ajouter des métadonnées
                for d in doc:
                    d.metadata['source_type'] = 'general_info'
                    d.metadata['filename'] = pdf_file.name
                docs.extend(doc)
                print(f"✅ Chargé: {pdf_file.name}")
            except Exception as e:
                print(f"❌ Erreur avec {pdf_file.name}: {e}")

        return docs

    def load_all_documents(self) -> List:
        """
        Charge tous les documents (HTML + PDF)

        Returns:
            Liste complète de tous les documents
        """
        print("📚 Chargement des documents HTML...")
        html_docs = self.load_html_files("concours_html")

        print("\n📄 Chargement des documents PDF...")
        pdf_docs = self.load_pdf_files(".")

        all_docs = html_docs + pdf_docs
        print(f"\n✅ Total: {len(all_docs)} documents chargés")

        return all_docs

# Fonction de démonstration avec des documents d'exemple
def create_sample_documents():
    """Crée des documents d'exemple si les vrais documents ne sont pas disponibles"""
    from langchain.schema import Document

    sample_docs = [
        Document(
            page_content="""
            Concours Ingénieur de Recherche en Informatique - Référence: IR2025-001
            Localisation: Toulouse, CNRS Occitanie-Ouest

            Mission: Développement et administration de systèmes de calcul haute performance
            pour les laboratoires de recherche en mathématiques et informatique.

            Profil recherché:
            - Formation: Bac+5 en informatique ou équivalent
            - Compétences: Linux, virtualisation, réseaux, sécurité
            - Expérience: 2 ans minimum en administration système

            Déroulement du concours:
            - Admissibilité: Étude du dossier (mars 2025)
            - Admission: Audition (mai 2025 à Paris)

            Rémunération: Selon grille IR CNRS, entre 2500€ et 4000€ brut mensuel
            """,
            metadata={"source_type": "concours", "filename": "ir_informatique_toulouse.html"}
        ),
        Document(
            page_content="""
            Concours Ingénieur d'Étude en Biologie - Référence: IE2025-042
            Localisation: Montpellier, Institut des Sciences Biologiques

            Mission: Support technique pour les équipes de recherche en biologie moléculaire.
            Gestion des équipements de laboratoire, cultures cellulaires, analyses.

            Profil recherché:
            - Formation: Bac+3 à Bac+5 en biologie
            - Compétences: Techniques de biologie moléculaire, PCR, culture cellulaire
            - Expérience: Débutants acceptés

            Déroulement du concours:
            - Admissibilité: QCM et étude de dossier (février 2025)
            - Admission: Épreuve pratique et audition (avril 2025 à Montpellier)

            Rémunération: Selon grille IE CNRS, environ 2000€ à 2800€ brut mensuel
            """,
            metadata={"source_type": "concours", "filename": "ie_biologie_montpellier.html"}
        ),
        Document(
            page_content="""
            Les carrières d'ingénieur au CNRS

            Le CNRS recrute chaque année plus de 170 ingénieurs dans des domaines très variés:
            - Sciences de l'information (informatique, réseaux, calcul)
            - Sciences de l'ingénieur (mécanique, électronique, instrumentation)
            - Sciences chimiques et des matériaux
            - Sciences de la vie
            - Sciences humaines et sociales (administration, communication, documentation)

            Différents grades:
            - Ingénieur de Recherche (IR): Niveau cadre supérieur, autonomie importante
            - Ingénieur d'Étude (IE): Niveau cadre, technicité élevée

            Avantages CNRS:
            - Sécurité de l'emploi (fonctionnaire)
            - Formations continues
            - Mobilité géographique possible
            - Participation à des projets de recherche d'excellence
            - RTT et congés généreux
            - Comité d'action sociale (réductions diverses)
            """,
            metadata={"source_type": "general_info", "filename": "carrieres_cnrs.pdf"}
        ),
        Document(
            page_content="""
            Déroulement des concours ingénieurs CNRS

            Phase 1 - Inscription:
            - Ouverture des inscriptions: généralement en janvier
            - Clôture: fin février
            - Inscription en ligne obligatoire sur le portail emploi CNRS

            Phase 2 - Admissibilité:
            - Étude des dossiers par un jury: mars-avril
            - Résultats publiés en ligne
            - Seuls les candidats admissibles passent aux épreuves d'admission

            Phase 3 - Admission:
            - Épreuves selon le concours: QCM, épreuves pratiques, audition
            - Période: avril à juin
            - Lieux: variables selon les concours (Paris, délégations régionales)

            Phase 4 - Résultats et nomination:
            - Publication liste principale et complémentaire: juin-juillet
            - Prise de fonction: selon les besoins, généralement septembre-décembre

            Conditions d'accès:
            - Nationalité française ou européenne
            - Diplôme requis selon le grade (Bac+3 minimum pour IE, Bac+5 pour IR)
            - Pas de limite d'âge (sauf pour certains concours spécifiques)
            - Jouissance des droits civiques
            """,
            metadata={"source_type": "general_info", "filename": "deroulement_concours.pdf"}
        )
    ]

    print("✅ Documents d'exemple créés (4 documents)")
    return sample_docs

# =====================================================================
# SECTION 5: CRÉATION DU SYSTÈME RAG
# =====================================================================

class RAGSystem:
    """Système RAG (Retrieval Augmented Generation) pour le chatbot CNRS"""

    def __init__(self, documents: List, model_name: str = "gpt-3.5-turbo"):
        """
        Initialise le système RAG

        Args:
            documents: Liste des documents à indexer
            model_name: Nom du modèle LLM à utiliser
        """
        self.documents = documents
        self.model_name = model_name
        self.vectorstore = None
        self.qa_chain = None

    def create_vectorstore(self, chunk_size: int = 1000, chunk_overlap: int = 200):
        """
        Crée la base vectorielle à partir des documents

        Args:
            chunk_size: Taille des chunks de texte
            chunk_overlap: Chevauchement entre chunks
        """
        print("📝 Découpage des documents en chunks...")

        # Découpage des documents
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", " ", ""]
        )

        splits = text_splitter.split_documents(self.documents)
        print(f"✅ {len(splits)} chunks créés")

        # Création des embeddings (utilise un modèle gratuit et efficace)
        print("🔄 Création des embeddings...")
        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )

        # Création de la base vectorielle
        print("💾 Indexation dans ChromaDB...")
        self.vectorstore = Chroma.from_documents(
            documents=splits,
            embedding=embeddings,
            collection_name="cnrs_concours"
        )

        print("✅ Base vectorielle créée!")

    def create_qa_chain(self, k: int = 4):
        """
        Crée la chaîne de question-réponse

        Args:
            k: Nombre de documents à récupérer pour chaque question
        """
        if not self.vectorstore:
            raise ValueError("Vous devez d'abord créer la vectorstore avec create_vectorstore()")

        # Sélection du modèle LLM
        if "gpt" in self.model_name.lower():
            if not OPENAI_AVAILABLE:
                raise ValueError("OpenAI n'est pas disponible. Installez: pip install langchain-openai")
            llm = ChatOpenAI(model=self.model_name, temperature=0)

        elif "claude" in self.model_name.lower():
            if not ANTHROPIC_AVAILABLE:
                raise ValueError("Anthropic n'est pas disponible. Installez: pip install langchain-anthropic")
            llm = ChatAnthropic(model=self.model_name, temperature=0)

        else:
            raise ValueError(f"Modèle non supporté: {self.model_name}")

        # Template de prompt personnalisé pour le CNRS
        template = """Tu es un assistant spécialisé dans les concours ingénieurs du CNRS.
        Ta mission est de répondre aux questions des candidats en te basant UNIQUEMENT sur les documents fournis.

        Règles importantes:
        1. Si l'information n'est pas dans les documents, réponds: "Je n'ai pas cette information dans ma base de documents."
        2. Cite toujours la source de tes informations (nom du document)
        3. Sois précis et factuel
        4. Ne jamais inventer d'informations

        Contexte (documents pertinents):
        {context}

        Question: {question}

        Réponse détaillée avec sources:"""

        PROMPT = PromptTemplate(
            template=template,
            input_variables=["context", "question"]
        )

        # Création de la chaîne QA
        self.qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=self.vectorstore.as_retriever(search_kwargs={"k": k}),
            chain_type_kwargs={"prompt": PROMPT},
            return_source_documents=True
        )

        print(f"✅ Chaîne QA créée avec le modèle {self.model_name}")

    def query(self, question: str) -> Dict[str, Any]:
        """
        Pose une question au système

        Args:
            question: La question à poser

        Returns:
            Dictionnaire contenant la réponse et les documents sources
        """
        if not self.qa_chain:
            raise ValueError("Vous devez d'abord créer la chaîne QA avec create_qa_chain()")

        result = self.qa_chain.invoke({"query": question})

        return {
            "question": question,
            "answer": result["result"],
            "source_documents": result["source_documents"]
        }

# =====================================================================
# SECTION 6: CRÉATION DU DATASET DE TEST
# =====================================================================

class TestDataset:
    """Classe pour créer et gérer le dataset de test"""

    def __init__(self):
        self.questions = []
        self.ground_truths = []
        self.question_types = []

    def add_question(self, question: str, ground_truth: str, q_type: str):
        """
        Ajoute une question au dataset

        Args:
            question: La question
            ground_truth: La réponse de référence
            q_type: Type de question (concours_specific, general_career, process, out_of_scope)
        """
        self.questions.append(question)
        self.ground_truths.append(ground_truth)
        self.question_types.append(q_type)

    def create_default_dataset(self):
        """Crée un dataset de test par défaut"""

        # Questions sur des concours spécifiques
        self.add_question(
            "Quel est le salaire d'un ingénieur de recherche en informatique?",
            "Entre 2500€ et 4000€ brut mensuel selon la grille IR CNRS",
            "concours_specific"
        )

        self.add_question(
            "Quelles compétences sont requises pour le poste d'ingénieur de recherche en informatique à Toulouse?",
            "Linux, virtualisation, réseaux, sécurité, avec un Bac+5 en informatique et 2 ans d'expérience minimum",
            "concours_specific"
        )

        self.add_question(
            "Où se déroulent les auditions pour le concours IE en biologie à Montpellier?",
            "Les auditions se déroulent à Montpellier en avril 2025",
            "concours_specific"
        )

        # Questions générales sur les carrières
        self.add_question(
            "Quels sont les avantages de travailler au CNRS?",
            "Sécurité de l'emploi (fonctionnaire), formations continues, mobilité géographique, projets d'excellence, RTT et congés généreux, comité d'action sociale",
            "general_career"
        )

        self.add_question(
            "Quelle est la différence entre un ingénieur de recherche et un ingénieur d'étude?",
            "L'ingénieur de recherche (IR) est de niveau cadre supérieur avec une grande autonomie, l'ingénieur d'étude (IE) est de niveau cadre avec une technicité élevée",
            "general_career"
        )

        self.add_question(
            "Combien de concours ingénieurs le CNRS organise-t-il par an?",
            "Plus de 170 concours par an dans des domaines très variés",
            "general_career"
        )

        # Questions sur le processus de concours
        self.add_question(
            "Quand s'inscrire aux concours CNRS?",
            "Les inscriptions ouvrent généralement en janvier et se clôturent fin février, en ligne sur le portail emploi CNRS",
            "process"
        )

        self.add_question(
            "Quelles sont les phases d'un concours CNRS?",
            "Phase 1: Inscription (janvier-février), Phase 2: Admissibilité avec étude des dossiers (mars-avril), Phase 3: Admission avec épreuves (avril-juin), Phase 4: Résultats et nomination (juin-juillet)",
            "process"
        )

        self.add_question(
            "Quelles sont les conditions pour passer un concours ingénieur CNRS?",
            "Nationalité française ou européenne, diplôme requis (Bac+3 pour IE, Bac+5 pour IR), jouissance des droits civiques, pas de limite d'âge pour la plupart",
            "process"
        )

        # Questions hors périmètre (le modèle doit refuser de répondre)
        self.add_question(
            "Quel est le salaire d'un ingénieur chez Google?",
            "Cette information ne concerne pas les concours CNRS et n'est pas disponible dans nos documents",
            "out_of_scope"
        )

        self.add_question(
            "Comment devenir ingénieur à l'INRIA?",
            "Cette question concerne l'INRIA, pas le CNRS. Je ne dispose pas d'informations sur les recrutements à l'INRIA",
            "out_of_scope"
        )

        self.add_question(
            "Quelle est la météo à Toulouse demain?",
            "Cette question n'a aucun rapport avec les concours ingénieurs du CNRS",
            "out_of_scope"
        )

        print(f"✅ Dataset créé: {len(self.questions)} questions")
        print(f"   - Concours spécifiques: {self.question_types.count('concours_specific')}")
        print(f"   - Carrières générales: {self.question_types.count('general_career')}")
        print(f"   - Processus concours: {self.question_types.count('process')}")
        print(f"   - Hors périmètre: {self.question_types.count('out_of_scope')}")

    def to_dict(self):
        """Convertit le dataset en dictionnaire"""
        return {
            "question": self.questions,
            "ground_truth": self.ground_truths,
            "question_type": self.question_types
        }

    def save_to_json(self, filepath: str):
        """Sauvegarde le dataset en JSON"""
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(self.to_dict(), f, ensure_ascii=False, indent=2)
        print(f"✅ Dataset sauvegardé: {filepath}")

# =====================================================================
# SECTION 7: ÉVALUATION AVEC RAGAS
# =====================================================================

class RAGEvaluator:
    """Classe pour évaluer un système RAG avec plusieurs métriques"""

    def __init__(self, rag_system: RAGSystem, test_dataset: TestDataset):
        """
        Initialise l'évaluateur

        Args:
            rag_system: Le système RAG à évaluer
            test_dataset: Le dataset de test
        """
        self.rag_system = rag_system
        self.test_dataset = test_dataset
        self.results = []

    def run_evaluation(self) -> pd.DataFrame:
        """
        Execute l'évaluation complète

        Returns:
            DataFrame avec tous les résultats
        """
        print(f"🔄 Évaluation en cours du modèle {self.rag_system.model_name}...")
        print(f"   Nombre de questions: {len(self.test_dataset.questions)}")

        # Collecte des réponses et contextes
        questions = []
        answers = []
        contexts = []
        ground_truths = []

        for i, question in enumerate(self.test_dataset.questions):
            print(f"   Question {i+1}/{len(self.test_dataset.questions)}...", end="\r")

            # Obtenir la réponse du système
            result = self.rag_system.query(question)

            # Préparer les données pour RAGAS
            questions.append(question)
            answers.append(result["answer"])

            # RAGAS attend une liste de contextes pour chaque question
            contexts.append([doc.page_content for doc in result["source_documents"]])

            ground_truths.append(self.test_dataset.ground_truths[i])

        print("\n✅ Toutes les réponses collectées")

        # Créer un DataFrame de base
        results_df = pd.DataFrame({
            "question": questions,
            "answer": answers,
            "ground_truth": ground_truths,
            "model": self.rag_system.model_name,
            "question_type": self.test_dataset.question_types
        })

        # Essayer d'utiliser RAGAS si disponible
        if RAGAS_AVAILABLE:
            print("🔄 Calcul des métriques RAGAS...")

            try:
                # Créer le dataset pour RAGAS
                eval_dataset = Dataset.from_dict({
                    "question": questions,
                    "answer": answers,
                    "contexts": contexts,
                    "ground_truth": ground_truths
                })

                ragas_results = evaluate(
                    eval_dataset,
                    metrics=[
                        faithfulness,
                        answer_relevancy,
                        context_precision,
                        context_recall
                    ]
                )

                print("✅ Métriques RAGAS calculées")

                # Ajouter les métriques RAGAS au DataFrame
                ragas_df = ragas_results.to_pandas()

                for col in ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']:
                    if col in ragas_df.columns:
                        results_df[col] = ragas_df[col]

            except Exception as e:
                print(f"⚠️ Erreur lors de l'évaluation RAGAS: {e}")
                print("   Continuons avec les métriques de base...")
        else:
            print("ℹ️  RAGAS non disponible - évaluation avec métriques basiques uniquement")

        return results_df

    def calculate_custom_metrics(self, results_df: pd.DataFrame) -> Dict:
        """
        Calcule des métriques personnalisées supplémentaires

        Args:
            results_df: DataFrame avec les résultats

        Returns:
            Dictionnaire avec les métriques
        """
        metrics = {
            "model": self.rag_system.model_name,
            "total_questions": len(results_df)
        }

        # Métriques RAGAS (si disponibles)
        if "faithfulness" in results_df.columns:
            metrics["faithfulness_mean"] = results_df["faithfulness"].mean()
            metrics["faithfulness_std"] = results_df["faithfulness"].std()

        if "answer_relevancy" in results_df.columns:
            metrics["answer_relevancy_mean"] = results_df["answer_relevancy"].mean()
            metrics["answer_relevancy_std"] = results_df["answer_relevancy"].std()

        if "context_precision" in results_df.columns:
            metrics["context_precision_mean"] = results_df["context_precision"].mean()

        if "context_recall" in results_df.columns:
            metrics["context_recall_mean"] = results_df["context_recall"].mean()

        # Détection des questions hors périmètre
        out_of_scope_df = results_df[results_df["question_type"] == "out_of_scope"]

        if len(out_of_scope_df) > 0:
            # Compter combien de fois le modèle refuse correctement
            refusal_keywords = ["n'ai pas", "ne dispose pas", "pas dans", "pas disponible",
                              "ne concerne pas", "hors", "information non disponible"]

            correct_refusals = 0
            for answer in out_of_scope_df["answer"]:
                if any(keyword in answer.lower() for keyword in refusal_keywords):
                    correct_refusals += 1

            metrics["out_of_scope_detection_rate"] = correct_refusals / len(out_of_scope_df)

        return metrics

# =====================================================================
# SECTION 8: COMPARAISON DE MODÈLES
# =====================================================================

class ModelComparison:
    """Classe pour comparer plusieurs modèles LLM"""

    def __init__(self, documents: List, test_dataset: TestDataset):
        """
        Initialise le comparateur

        Args:
            documents: Documents à indexer
            test_dataset: Dataset de test
        """
        self.documents = documents
        self.test_dataset = test_dataset
        self.all_results = []
        self.all_metrics = []

    def evaluate_model(self, model_name: str) -> tuple:
        """
        Évalue un modèle spécifique

        Args:
            model_name: Nom du modèle à évaluer

        Returns:
            Tuple (results_df, metrics_dict)
        """
        print(f"\n{'='*70}")
        print(f"🤖 ÉVALUATION DU MODÈLE: {model_name}")
        print(f"{'='*70}\n")

        # Créer le système RAG
        rag = RAGSystem(self.documents, model_name=model_name)
        rag.create_vectorstore()
        rag.create_qa_chain()

        # Évaluer
        evaluator = RAGEvaluator(rag, self.test_dataset)
        results_df = evaluator.run_evaluation()
        metrics = evaluator.calculate_custom_metrics(results_df)

        # Sauvegarder
        self.all_results.append(results_df)
        self.all_metrics.append(metrics)

        # Afficher les résultats
        print(f"\n📊 RÉSULTATS POUR {model_name}:")
        print("-" * 50)
        for key, value in metrics.items():
            if isinstance(value, float):
                print(f"  {key}: {value:.3f}")
            else:
                print(f"  {key}: {value}")

        return results_df, metrics

    def compare_all_models(self, model_list: List[str]):
        """
        Compare plusieurs modèles

        Args:
            model_list: Liste des noms de modèles à comparer
        """
        for model_name in model_list:
            try:
                self.evaluate_model(model_name)
            except Exception as e:
                print(f"❌ Erreur avec {model_name}: {e}")
                continue

    def create_comparison_report(self) -> pd.DataFrame:
        """
        Crée un rapport comparatif

        Returns:
            DataFrame avec la comparaison des modèles
        """
        if not self.all_metrics:
            print("⚠️ Aucun résultat disponible")
            return pd.DataFrame()

        comparison_df = pd.DataFrame(self.all_metrics)

        print("\n" + "="*70)
        print("📊 TABLEAU COMPARATIF DES MODÈLES")
        print("="*70 + "\n")
        print(comparison_df.to_string(index=False))

        return comparison_df

    def plot_comparison(self):
        """Crée des visualisations comparatives"""
        if not self.all_metrics:
            print("⚠️ Aucun résultat à visualiser")
            return

        comparison_df = pd.DataFrame(self.all_metrics)

        # Graphique 1: Comparaison des métriques principales
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle('Comparaison des Performances des Modèles', fontsize=16, fontweight='bold')

        metrics_to_plot = [
            ('faithfulness_mean', 'Fidélité (Faithfulness)'),
            ('answer_relevancy_mean', 'Pertinence (Answer Relevancy)'),
            ('context_precision_mean', 'Précision du Contexte'),
            ('out_of_scope_detection_rate', 'Détection Hors Périmètre')
        ]

        for idx, (metric, title) in enumerate(metrics_to_plot):
            ax = axes[idx // 2, idx % 2]

            if metric in comparison_df.columns:
                comparison_df.plot(
                    x='model',
                    y=metric,
                    kind='bar',
                    ax=ax,
                    color='steelblue',
                    legend=False
                )
                ax.set_title(title, fontweight='bold')
                ax.set_xlabel('Modèle')
                ax.set_ylabel('Score')
                ax.set_ylim(0, 1.1)
                ax.grid(axis='y', alpha=0.3)

                # Ajouter les valeurs sur les barres
                for container in ax.containers:
                    ax.bar_label(container, fmt='%.3f')
            else:
                ax.text(0.5, 0.5, f'Métrique\n{metric}\nnon disponible',
                       ha='center', va='center', transform=ax.transAxes)
                ax.set_title(title, fontweight='bold')

        plt.tight_layout()
        plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
        print("✅ Graphique sauvegardé: model_comparison.png")
        plt.show()

        # Graphique 2: Radar chart pour comparaison globale
        if len(self.all_metrics) > 0:
            self.plot_radar_chart(comparison_df)

    def plot_radar_chart(self, comparison_df: pd.DataFrame):
        """Crée un radar chart pour comparaison visuelle"""
        import math

        metrics = ['faithfulness_mean', 'answer_relevancy_mean',
                  'context_precision_mean', 'out_of_scope_detection_rate']

        # Vérifier que les métriques existent
        available_metrics = [m for m in metrics if m in comparison_df.columns]

        if len(available_metrics) < 3:
            print("⚠️ Pas assez de métriques pour le radar chart")
            return

        # Nombre de variables
        num_vars = len(available_metrics)

        # Calculer l'angle pour chaque axe
        angles = [n / float(num_vars) * 2 * math.pi for n in range(num_vars)]
        angles += angles[:1]

        # Créer le plot
        fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

        # Plot pour chaque modèle
        for idx, row in comparison_df.iterrows():
            values = [row[m] for m in available_metrics]
            values += values[:1]

            ax.plot(angles, values, 'o-', linewidth=2, label=row['model'])
            ax.fill(angles, values, alpha=0.15)

        # Labels
        labels = [m.replace('_mean', '').replace('_', ' ').title()
                 for m in available_metrics]
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(labels, size=10)

        # Limites
        ax.set_ylim(0, 1)
        ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
        ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'])
        ax.grid(True)

        plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
        plt.title('Comparaison Globale des Modèles\n(Radar Chart)',
                 size=14, fontweight='bold', pad=20)

        plt.tight_layout()
        plt.savefig('radar_comparison.png', dpi=300, bbox_inches='tight')
        print("✅ Radar chart sauvegardé: radar_comparison.png")
        plt.show()

    def export_detailed_results(self, filepath: str = "detailed_results.xlsx"):
        """
        Exporte les résultats détaillés dans Excel

        Args:
            filepath: Chemin du fichier Excel
        """
        if not self.all_results:
            print("⚠️ Aucun résultat à exporter")
            return

        with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
            # Onglet 1: Résumé comparatif
            comparison_df = pd.DataFrame(self.all_metrics)
            comparison_df.to_excel(writer, sheet_name='Comparaison', index=False)

            # Onglet pour chaque modèle
            for idx, results_df in enumerate(self.all_results):
                model_name = results_df['model'].iloc[0]
                sheet_name = f"Model_{idx+1}_{model_name[:20]}"
                results_df.to_excel(writer, sheet_name=sheet_name, index=False)

        print(f"✅ Résultats détaillés exportés: {filepath}")

# =====================================================================
# SECTION 9: EXEMPLE D'UTILISATION COMPLET
# =====================================================================

def main_evaluation_pipeline():
    """
    Pipeline complet d'évaluation
    À exécuter dans Google Colab
    """

    print("="*70)
    print("🚀 DÉMARRAGE DU PIPELINE D'ÉVALUATION")
    print("="*70)

    # Vérification des API
    if not API_CONFIGURED:
        print("\n❌ Configuration API requise!")
        print("Ajoutez vos clés API et relancez ce bloc.")
        return None, None

    # Étape 1: Charger les documents
    print("\n📚 ÉTAPE 1: Chargement des documents")
    print("-" * 70)

    # Option A: Charger les vrais documents (si disponibles)
    try:
        loader = DocumentLoader("./data")
        documents = loader.load_all_documents()

        if len(documents) == 0:
            raise ValueError("Aucun document trouvé")

    except Exception as e:
        print(f"⚠️ Impossible de charger les documents réels: {e}")
        print("📝 Utilisation des documents d'exemple à la place")
        documents = create_sample_documents()

    # Étape 2: Créer le dataset de test
    print("\n❓ ÉTAPE 2: Création du dataset de test")
    print("-" * 70)

    test_dataset = TestDataset()
    test_dataset.create_default_dataset()

    # Sauvegarder le dataset
    test_dataset.save_to_json("test_dataset.json")

    # Étape 3: Définir les modèles à comparer
    print("\n🤖 ÉTAPE 3: Configuration des modèles à évaluer")
    print("-" * 70)

    # Liste des modèles disponibles
    models_to_test = []

    if OPENAI_AVAILABLE and os.getenv("OPENAI_API_KEY"):
        models_to_test.extend([
            "gpt-3.5-turbo",
            "gpt-4",
        ])
        print("✅ Modèles OpenAI ajoutés")

    if ANTHROPIC_AVAILABLE and os.getenv("ANTHROPIC_API_KEY"):
        models_to_test.extend([
            "claude-3-haiku-20240307",
            "claude-3-sonnet-20240229",
        ])
        print("✅ Modèles Anthropic ajoutés")

    if not models_to_test:
        print("❌ Aucun modèle disponible!")
        print("ℹ️  Vérifiez vos clés API et les installations")
        return None, None

    print(f"\n📋 Modèles sélectionnés pour la comparaison:")
    for i, model in enumerate(models_to_test, 1):
        print(f"   {i}. {model}")

    # Étape 4: Lancer l'évaluation comparative
    print("\n⚙️ ÉTAPE 4: Évaluation des modèles")
    print("-" * 70)

    comparator = ModelComparison(documents, test_dataset)
    comparator.compare_all_models(models_to_test)

    # Étape 5: Générer les rapports
    print("\n📊 ÉTAPE 5: Génération des rapports")
    print("-" * 70)

    comparison_report = comparator.create_comparison_report()

    # Étape 6: Créer les visualisations
    print("\n📈 ÉTAPE 6: Création des visualisations")
    print("-" * 70)

    comparator.plot_comparison()

    # Étape 7: Exporter les résultats
    print("\n💾 ÉTAPE 7: Export des résultats")
    print("-" * 70)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    comparator.export_detailed_results(f"evaluation_results_{timestamp}.xlsx")

    print("\n" + "="*70)
    print("✅ ÉVALUATION TERMINÉE AVEC SUCCÈS!")
    print("="*70)

    return comparator, comparison_report

# =====================================================================
# SECTION 10: FONCTIONS UTILITAIRES SUPPLÉMENTAIRES
# =====================================================================

def test_single_model_interactive(documents, model_name="gpt-3.5-turbo"):
    """
    Test interactif d'un seul modèle
    Utile pour tester rapidement un modèle avant l'évaluation complète

    Args:
        documents: Les documents à utiliser
        model_name: Nom du modèle à tester
    """
    print(f"🧪 Test interactif du modèle: {model_name}")

    # Créer le système RAG
    rag = RAGSystem(documents, model_name=model_name)
    rag.create_vectorstore()
    rag.create_qa_chain()

    # Questions de test rapide
    test_questions = [
        "Combien le CNRS recrute-t-il d'ingénieurs par an?",
        "Quels sont les avantages de travailler au CNRS?",
        "Quel est le salaire d'un ingénieur chez Microsoft?"  # Hors périmètre
    ]

    for i, question in enumerate(test_questions, 1):
        print(f"\n{'='*70}")
        print(f"Question {i}: {question}")
        print('-'*70)

        result = rag.query(question)
        print(f"Réponse: {result['answer']}")
        print(f"\nNombre de documents sources: {len(result['source_documents'])}")

    print(f"\n{'='*70}")
    print("✅ Test terminé")

    return rag

def analyze_failure_cases(results_df: pd.DataFrame, threshold: float = 0.5):
    """
    Analyse les cas d'échec (faible score)

    Args:
        results_df: DataFrame avec les résultats
        threshold: Seuil en dessous duquel on considère un échec
    """
    print("\n🔍 ANALYSE DES CAS D'ÉCHEC")
    print("="*70)

    if "faithfulness" not in results_df.columns:
        print("⚠️ Métrique faithfulness non disponible")
        return

    failures = results_df[results_df["faithfulness"] < threshold]

    if len(failures) == 0:
        print(f"✅ Aucun cas d'échec (tous les scores > {threshold})")
        return

    print(f"❌ {len(failures)} cas avec faithfulness < {threshold}")
    print()

    for idx, row in failures.iterrows():
        print(f"Question: {row['question']}")
        print(f"Réponse: {row['answer'][:200]}...")
        print(f"Score faithfulness: {row['faithfulness']:.3f}")
        print(f"Type: {row['question_type']}")
        print("-" * 70)

# =====================================================================
# INSTRUCTIONS D'UTILISATION
# =====================================================================

"""
═══════════════════════════════════════════════════════════════════════
📖 GUIDE D'UTILISATION RAPIDE
═══════════════════════════════════════════════════════════════════════

1️⃣ CONFIGURATION INITIALE (à faire une fois)
   - Configurer les clés API dans la Section 3
   - Uploader vos documents dans Google Colab (ou utiliser les exemples)

2️⃣ EXÉCUTION RAPIDE (pour une évaluation complète)

   # Lancer le pipeline complet
   comparator, report = main_evaluation_pipeline()

3️⃣ TEST RAPIDE D'UN SEUL MODÈLE

   # Charger les documents
   documents = create_sample_documents()

   # Tester un modèle
   rag = test_single_model_interactive(documents, "gpt-3.5-turbo")

4️⃣ ÉVALUATION PERSONNALISÉE

   # Créer votre propre dataset de test
   my_dataset = TestDataset()
   my_dataset.add_question(
       "Ma question",
       "Réponse attendue",
       "question_type"
   )

   # Évaluer un modèle spécifique
   documents = create_sample_documents()
   rag = RAGSystem(documents, model_name="gpt-4")
   rag.create_vectorstore()
   rag.create_qa_chain()

   evaluator = RAGEvaluator(rag, my_dataset)
   results = evaluator.run_evaluation()

5️⃣ ANALYSER LES RÉSULTATS

   # Voir les cas d'échec
   analyze_failure_cases(results, threshold=0.5)

   # Exporter vers Excel
   comparator.export_detailed_results("mes_resultats.xlsx")

═══════════════════════════════════════════════════════════════════════
📊 MÉTRIQUES EXPLIQUÉES
═══════════════════════════════════════════════════════════════════════

• Faithfulness (0-1): Le modèle invente-t-il des informations?
  → Plus c'est proche de 1, mieux c'est
  → Crucial pour votre projet CNRS

• Answer Relevancy (0-1): La réponse correspond-elle à la question?
  → Plus c'est proche de 1, mieux c'est

• Context Precision (0-1): Les bons documents sont-ils en haut?
  → Plus c'est proche de 1, mieux c'est

• Context Recall (0-1): Tous les documents nécessaires sont récupérés?
  → Plus c'est proche de 1, mieux c'est

• Out of Scope Detection (0-1): Détecte-t-il les questions hors sujet?
  → Plus c'est proche de 1, mieux c'est
  → Crucial pour ne pas donner de fausses infos

═══════════════════════════════════════════════════════════════════════
"""

print(__doc__)

# Task
Uninstall the existing `protobuf` package, then reinstall all dependencies in the notebook's code cell to resolve the `VersionError` and ensure compatibility.

## Uninstall conflicting protobuf

### Subtask:
Explicitly uninstall the `protobuf` package to ensure a clean slate, removing any conflicting versions.


**Reasoning**:
The subtask requires uninstalling the `protobuf` package. I will use the `!pip uninstall protobuf -y` command to achieve this.



In [12]:
print("Uninstalling existing protobuf package...")
!pip uninstall protobuf -y
print("✓ protobuf package uninstalled.")

Uninstalling existing protobuf package...
Found existing installation: protobuf 6.33.3
Uninstalling protobuf-6.33.3:
  Successfully uninstalled protobuf-6.33.3
✓ protobuf package uninstalled.


## Update all dependencies

### Subtask:
Reinstall all required libraries, allowing `pip` to resolve and install a compatible `protobuf` version alongside `ragas` and other dependencies.


**Reasoning**:
The subtask requires reinstalling all dependencies. I will extract and execute the `!pip install` commands from the specified cell (`qwiOCZr7RnaQ`) to ensure all libraries, including a compatible `protobuf` version, are installed.



In [13]:
print("Reinstalling all required libraries...")

# Installation avec versions spécifiques pour éviter les conflits
!pip install -q langchain==0.1.0
!pip install -q langchain-community==0.0.13
!pip install -q langchain-openai==0.0.2
!pip install -q langchain-anthropic==0.1.1
!pip install -q chromadb==0.4.22
!pip install -q sentence-transformers==2.2.2
!pip install -q ragas==0.1.4
!pip install -q datasets==2.16.1
!pip install -q pypdf==3.17.4
!pip install -q beautifulsoup4==4.12.3
!pip install -q html2text==2020.1.16
!pip install -q unstructured==0.11.8
!pip install -q openai==1.7.2
!pip install -q anthropic==0.8.1
!pip install -q pandas matplotlib seaborn plotly
!pip install -q openpyxl

print("✓ All required libraries reinstalled.")

Reinstalling all required libraries...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
onnx 1.20.1 requires protobuf>=4.25.1, which is not installed.
onnxruntime 1.23.2 requires protobuf, which is not installed.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, which is not installed.
orbax-checkpoint 0.11.31 requires protobuf, which is not installed.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, which is not installed.
google-cloud-aiplatform 1.130.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, which is not installed.
wandb 0.23.1 requires protobuf!=4.21.0,!=5.28.0,<7,>=3.19.0; python_version > "3.9" and sys_platform == "linux", which is not installed.
tensorboard 2.19.0 requires protobuf!=4.24.0,>=3.19.6, which is not installed.
tensorflow-datasets 4.9.9 requi

## Re-execute the cell

### Subtask:
Run the modified code cell to apply the installation changes and verify that the `VersionError` is resolved.


## Summary:

### Data Analysis Key Findings
*   The `protobuf` package (version `6.33.3`) was successfully uninstalled to clear potential version conflicts.
*   All specified libraries, including `langchain==0.1.0`, `ragas==0.1.4`, and `openai==1.7.2`, were reinstalled.
*   Despite the reinstallation, numerous `pip` dependency conflicts were reported, indicating ongoing version incompatibilities.
*   Specific conflicts include:
    *   `onnx` requiring `protobuf>=4.25.1`.
    *   `tensorflow` requiring `protobuf!=4.21.0,...<6.0.0dev,>=3.20.3`.
    *   `google-colab` requiring `pandas==2.2.2`, while `pandas 2.3.3` was installed.
    *   `instructor` requiring `openai<3.0.0,>=2.0.0`, while `openai 1.7.2` was installed.

### Insights or Next Steps
*   The initial `VersionError` related to `protobuf` might not be fully resolved, as the reinstallation process introduced or highlighted other significant dependency conflicts across various packages.
*   A subsequent step should involve addressing these newly reported dependency conflicts, potentially by adjusting the specified versions of conflicting packages or utilizing a more robust dependency management strategy to ensure a stable environment.
